In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import gradio as gr
import langchain as lc
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

In [ ]:
data = pd.read_csv(r'C:\EY\Financial-Agent\data\all_stocks.csv')
data.head()

In [ ]:
print(f'The dataframe has {data.shape[0]} rows and {data.shape[1]} columns.')
print(f'Columns: {data.columns.tolist()}')

In [ ]:
# Parse dates, sort chronologically, and build the classification label

df = data.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by=['Date', 'ticker']).reset_index(drop=True)

df['direction'] = (df['return_1d'] > 0).astype(int)

feature_columns = [
    'open', 'high', 'low', 'volume',
    'return_5d', 'ma_20', 'ma_50', 'rsi_14',
    'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_mid', 'bb_lower',
    'volatility_20'
]

df = df[feature_columns + ['close', 'direction']].dropna().reset_index(drop=True)

X = df[feature_columns]
y_reg = df['close']
y_clf = df['direction']

print('Prepared dataset shape:', df.shape)
print('Feature columns:', feature_columns)
print('Classification label distribution:')
print(df['direction'].value_counts(normalize=True).rename('proportion'))

In [ ]:
# Split chronologically for train / validation / test
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train_reg, y_val_reg, y_test_reg = y_reg.iloc[:train_end], y_reg.iloc[train_end:val_end], y_reg.iloc[val_end:]
y_train_clf, y_val_clf, y_test_clf = y_clf.iloc[:train_end], y_clf.iloc[train_end:val_end], y_clf.iloc[val_end:]

print('Train shape:', X_train.shape)
print('Validation shape:', X_val.shape)
print('Test shape:', X_test.shape)

In [ ]:
model_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')

model_regressor.fit(X_train, y_train_reg)
model_classifier.fit(X_train, y_train_clf)

def evaluate_regression(model, X, y, label):
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y, y_pred)
    print(f'{label} regression metrics:')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  MSE: {mse:.4f}')
    print(f'  R2: {r2:.4f}')
    return y_pred


def evaluate_classification(model, X, y, label):
    y_pred = model.predict(X)
    print(f'{label} classification metrics:')
    print(f'  Accuracy:  {accuracy_score(y, y_pred):.4f}')
    print(f'  Precision: {precision_score(y, y_pred, zero_division=0):.4f}')
    print(f'  Recall:    {recall_score(y, y_pred, zero_division=0):.4f}')
    print(f'  F1 score:  {f1_score(y, y_pred, zero_division=0):.4f}')
    print('  Confusion matrix:')
    print(confusion_matrix(y, y_pred))
    print('  Classification report:')
    print(classification_report(y, y_pred, zero_division=0))
    return y_pred

print('--- Regressor ---')
evaluate_regression(model_regressor, X_train, y_train_reg, 'Train')
evaluate_regression(model_regressor, X_val, y_val_reg, 'Validation')
evaluate_regression(model_regressor, X_test, y_test_reg, 'Test')

print('\n--- Classifier ---')
evaluate_classification(model_classifier, X_train, y_train_clf, 'Train')
evaluate_classification(model_classifier, X_val, y_val_clf, 'Validation')
evaluate_classification(model_classifier, X_test, y_test_clf, 'Test')